In [25]:
from langgraph.graph import StateGraph, END, START
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

 

In [26]:
generator_model = ChatOllama(model = "gemma3:latest")
evaluator_model = ChatOllama(model = "gemma3:latest")
optimiser_model = ChatOllama(model = "gemma3:latest")


In [27]:
class evaluate_structure(BaseModel):
    feedback : str = Field(..., description="feedback for the tweet.")
    evaluation : Literal["approved", "not approved"] = Field(..., description="Final evaluation result.")

In [28]:
structured_evaluator_model = evaluator_model.with_structured_output(evaluate_structure)

In [29]:
class PostState(TypedDict):
    topic : str
    post : str
    feedback : str
    evaluation : Literal["approved", "not approved"]
    iteration : int   
    max_iter: int

In [30]:
def generate(state:PostState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    result = generator_model.invoke(messages)
    return {"post" : result.content}

In [31]:
def evaluate(state:PostState):
     messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['post']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY with a valid JSON object using this exact schema:
{{
    "evaluation": "approved" or "not approved",
    "feedback": "One paragraph explaining the strengths and weaknesses"
}}
""")
]
     result = structured_evaluator_model.invoke(messages)

     return {"feedback" : result.feedback, "evaluation": result.evaluation}

In [32]:
def optimise(state:PostState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original post:
{state['post']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    result = optimiser_model.invoke(messages)
    iteration = state['iteration']+1
    return {"post":result.content, "iteration" : iteration}

In [33]:
def check(state:PostState):
    if state['evaluation'] == "approved" or state["i" \
    "teration"] >= state['max_iter']:
        return "approved"
    else:
        return "not approved"

In [34]:
graph = StateGraph(PostState)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("optimise", optimise)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", check, {"approved": END, "not approved": "optimise" })
graph.add_edge("optimise", "evaluate")


In [35]:
workflow=graph.compile()

In [36]:
initial_state = {
    "topic": "IPL",
    "iteration": 1,
    "max_iter": 5
}
result = workflow.invoke(initial_state)
result 

{'topic': 'IPL',
 'post': 'Okay, let’s go for maximum chaos! Here are a few options building on your initial draft, aiming for that viral edge:\n\n**Option 1 (Most Aggressive):**\n\n“IPL arguments = my brain attempting to process stats. It\'s pure, unadulterated SHOUTING. 🤯 Rule change needed IMMEDIATELY. Send wine & chaos. 🏏 #IPL #CricketMadness”\n\n**Option 2 (Slightly More Playful):**\n\n"My IPL debates are a full-blown warzone of numbers & opinions. 💥 Seriously considering calling in the riot police. Anyone else? 🏏🤯 #IPL #Cricket”\n\n**Option 3 (Shortest & Punchiest):**\n\n“IPL arguments: Statistical fury unleashed! 🤯 Someone just screamed about DRS. Send help (and a very large beer). 🍻🏏 #IPL #Cricket”\n\n---\n\n**Rationale for Changes:**\n\n*   **Stronger Verbs:**  I\'ve replaced "are basically" with stronger words like “attempting,” “full-blown warzone,” and “unleashed.”\n*   **Exaggerated Reactions:** The suggestions of riot police or a large beer heighten the absurdity.\n*   **